In [6]:
import (
	"bytes"
	"fmt"
	"math"

	_ "github.com/gomlx/gomlx/backends/default"

	"github.com/TuSKan/geocolor"
	"github.com/gomlx/gomlx/backends"
	"github.com/gomlx/gomlx/pkg/core/graph"
	"github.com/gomlx/gomlx/pkg/ml/context"

	"github.com/janpfeifer/gonb/gonbui"

	grob "github.com/MetalBlueberry/go-plotly/generated/v2.34.0/graph_objects"
	"github.com/MetalBlueberry/go-plotly/pkg/offline"
	"github.com/MetalBlueberry/go-plotly/pkg/types"

	// TODO: Replace with the actual local import path of your geocolor package
	"github.com/TuSKan/geocolor"
)

func main() {
	// 1. Initialize GoMLX backend and context
	backend, err := backends.New()
	if err != nil {
		fmt.Printf("Failed to initialize backend: %v\n", err)
		return
	}
	ctx := context.New()

	// 2. Define grid parameters
	resL, resA, resB := 4, 5, 5
	minBounds := [3]float64{0.0, -10.0, -10.0}
	maxBounds := [3]float64{100.0, 10.0, 10.0}
	originIndex := [3]int{0, 2, 2}

	// 3. Define and execute the GoMLX computational graph
	pipelineFn := func(ctx *context.Context, g *graph.Graph) *graph.Node {
		coords := geocolor.BuildCoordinateGridGraph(g, resL, resA, resB, minBounds, maxBounds)
		localMetrics := geocolor.ComputeLocalMetric(g, coords)
		distances := geocolor.SolveGlobalDistances(g, localMetrics, originIndex)
		
		cParam := graph.Const(g, float32(0.1))
		dampened := geocolor.ApplyDiminishingReturns(g, distances, cParam)
		
		neutralIndices := geocolor.ExtractNeutralAxis(g, dampened)
		return geocolor.ComputeGeometricColor(g, coords, neutralIndices, minBounds, maxBounds)
	}

	e, err := context.NewExec(backend, ctx, pipelineFn)
	if err != nil {
		fmt.Printf("Failed to compile GoMLX pipeline: %v\n", err)
		return
	}

	results, err := e.Exec()
	if err != nil {
		fmt.Printf("Failed to execute GoMLX pipeline: %v\n", err)
		return
	}

	// 4. Extract the 4D float32 tensor directly from memory (no binary files needed)
	vals := results[0].Value().([][][][]float32)

	// 5. Process the Neutral Axis
	neutral_L := make([]float64, resL)
	neutral_C := make([]float64, resL)

	for L := 0; L < resL; L++ {
		minC := float32(math.Inf(1))
		var targetL float32

		for a := 0; a < resA; a++ {
			for b := 0; b < resB; b++ {
				lVal := vals[L][a][b][0]
				cVal := vals[L][a][b][1]

				if cVal < minC {
					minC = cVal
					targetL = lVal
				}
			}
		}

		neutral_L[L] = float64(targetL)
		neutral_C[L] = float64(minC)
	}

	// 6. Plot the results using go-plotly
	fig := &grob.Fig{
		Data: []grob.Type{
			&grob.Scatter{
				X:    neutral_C,
				Y:    neutral_L,
				Mode: grob.ScatterModeMarkersLines,
				Marker: &grob.ScatterMarker{
					Color: grob.Color{String: "blue"},
				},
				Name: grob.String("Geometric Neutral Axis"),
			},
		},
		Layout: &grob.Layout{
			Title: &grob.LayoutTitle{
				Text: grob.String("Non-Riemannian Geometric Neutral Axis"),
			},
			Xaxis: &grob.LayoutXaxis{
				Title: &grob.LayoutXaxisTitle{
					Text: grob.String("Geometric Chroma (C_geo)"),
				},
			},
			Yaxis: &grob.LayoutYaxis{
				Title: &grob.LayoutYaxisTitle{
					Text: grob.String("Lightness (L)"),
				},
			},
			Shapes: []grob.LayoutShape{
				{
					Type: grob.LayoutShapeTypeLine,
					X0:   0, X1: 0,
					Y0:   0, Y1: 100, // Matches maxBounds[0]
					Line: &grob.LayoutShapeLine{
						Color: grob.Color{String: "black"},
						Dash:  grob.LayoutShapeLineDashDash,
					},
				},
			},
		},
	}

	// 7. Render directly into the GoNB output cell
	var buf bytes.Buffer
	err = offline.ToHtml(fig, &buf)
	if err != nil {
		fmt.Printf("Failed to render plot: %v\n", err)
		return
	}
	
	gonbui.DisplayHtml(buf.String())
}

# gonb_0f129393 

 

 Cell[6]: Line 92 
 ./main.go:87:16: undefined: grob.Type
 

	// 6. Plot the results using go-plotly
	fig := &grob.Fig{
 Data: []grob.Type{
 &grob.Scatter{
 X: neutral_C,

 

 

 Cell[6]: Line 94 
 ./main.go:89:11: cannot use neutral_C (variable of type []float64) as *"github.com/MetalBlueberry/go-plotly/pkg/types".DataArrayType value in struct literal
 
	fig := &grob.Fig{
 Data: []grob.Type{
 &grob.Scatter{
 X: neutral_C,
 Y: neutral_L,
 Mode: grob.ScatterModeMarkersLines,

 

 

 Cell[6]: Line 95 
 ./main.go:90:11: cannot use neutral_L (variable of type []float64) as *"github.com/MetalBlueberry/go-plotly/pkg/types".DataArrayType value in struct literal
 
 Data: []grob.Type{
 &grob.Scatter{
 X: neutral_C,
 Y: neutral_L,
 Mode: grob.ScatterModeMarkersLines,
 Marker: &grob.ScatterMarker{

 

 

 Cell[6]: Line 96 
 ./main.go:91:16: undefined: grob.ScatterModeMarkersLines
 
 &grob.Scatter{
 X: neutral_C,
 Y: neutral_L,
 Mode: grob.ScatterModeMarkersLines,
 Marker: &grob.ScatterMarker{
 Color: grob.Color{String: "blue"},

 

 

 Cell[6]: Line 98 
 ./main.go:93:18: undefined: grob.Color
 
 Y: neutral_L,
 Mode: grob.ScatterModeMarkersLines,
 Marker: &grob.ScatterMarker{
 Color: grob.Color{String: "blue"},
 },
 Name: grob.String("Geometric Neutral Axis"),

 

 

 Cell[6]: Line 100 
 ./main.go:95:16: undefined: grob.String
 
 Marker: &grob.ScatterMarker{
 Color: grob.Color{String: "blue"},
 },
 Name: grob.String("Geometric Neutral Axis"),
 },
 },

 

 

 Cell[6]: Line 105 
 ./main.go:100:16: undefined: grob.String
 
 },
 Layout: &grob.Layout{
 Title: &grob.LayoutTitle{
 Text: grob.String("Non-Riemannian Geometric Neutral Axis"),
 },
 Xaxis: &grob.LayoutXaxis{

 

 

 Cell[6]: Line 109 
 ./main.go:104:17: undefined: grob.String
 
 },
 Xaxis: &grob.LayoutXaxis{
 Title: &grob.LayoutXaxisTitle{
 Text: grob.String("Geometric Chroma (C_geo)"),
 },
 },

 

 

 Cell[6]: Line 114 
 ./main.go:109:17: undefined: grob.String
 
 },
 Yaxis: &grob.LayoutYaxis{
 Title: &grob.LayoutYaxisTitle{
 Text: grob.String("Lightness (L)"),
 },
 },

 

 

 Cell[6]: Line 123 
 ./main.go:118:19: undefined: grob.Color
 
 X0: 0, X1: 0,
 Y0: 0, Y1: 100, // Matches maxBounds[0]
 Line: &grob.LayoutShapeLine{
 Color: grob.Color{String: "black"},
 Dash: grob.LayoutShapeLineDashDash,
 },

 

 

 Cell[6]: Line 123 
 ./main.go:118:19: too many errors
 
 X0: 0, X1: 0,
 Y0: 0, Y1: 100, // Matches maxBounds[0]
 Line: &grob.LayoutShapeLine{
 Color: grob.Color{String: "black"},
 Dash: grob.LayoutShapeLineDashDash,
 },

ERROR: failed to run "/usr/bin/go build -o /tmp/gonb_0f129393/gonb_0f129393": exit status 1